# mithril2pq tables

Every table is a DuckDB view over its Parquet files, so nothing is read until a query runs.

- `tables.<name>` / `tables["<name>"]`: the whole table as a pandas DataFrame, read on first access and cached.
- `load(name, columns=..., where=..., limit=...)`: part of a table.
- `sql("select ...")`: any query over the views, as a DataFrame.

With `hex=True` (the default) hash, key and signature columns come back as lowercase hex strings. In SQL,
compare them as blobs (`where hash = from_hex('...')`) or convert with the `hx()` macro.
Large tables (`plutus_data_node`, `tx_output_asset`, `tx_input`, ...) can need tens of GB as a full DataFrame:
select columns, filter, or aggregate in SQL instead.

In [ ]:
import os
from collections.abc import Mapping
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

DATA = Path(os.environ.get("MITHRIL2PQ_DATA", "/data"))

con = duckdb.connect()
con.sql(f"SET memory_limit = '{os.environ.get('DUCKDB_MEMORY_LIMIT', '16GB')}'")
con.sql("CREATE OR REPLACE MACRO hx(b) AS lower(hex(b))")

TABLES = sorted(p.name for p in DATA.iterdir() if p.is_dir() and any(p.glob("part-*.parquet")))
# Columns stored as fixed-size binary: hashes, keys, signatures, IP addresses.
HASH_COLUMNS = {}
for _name in TABLES:
    con.sql(f"CREATE OR REPLACE VIEW {_name} AS SELECT * FROM read_parquet('{DATA / _name}/part-*.parquet')")
    _schema = pq.read_schema(next((DATA / _name).glob("part-*.parquet")))
    HASH_COLUMNS[_name] = {f.name for f in _schema if pa.types.is_fixed_size_binary(f.type)}


def _df(rel: duckdb.DuckDBPyRelation) -> pd.DataFrame:
    # Arrow-backed columns keep nullable uint64 values exact instead of turning them into float64.
    return rel.to_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)


def sql(query: str, *params) -> pd.DataFrame:
    """Run a query over the table views and return a DataFrame."""
    return _df(con.sql(query, params=list(params) or None))


def load(name: str, columns=None, where: str | None = None, limit: int | None = None, hex: bool = True) -> pd.DataFrame:
    """Read a table, or part of it.

    columns: list of column names; where: a SQL condition, e.g. "lovelace > 10e12";
    hex: return hash columns as hex strings rather than bytes.
    """
    if name not in HASH_COLUMNS:
        raise KeyError(name)
    cols = columns or [f.name for f in pq.read_schema(next((DATA / name).glob("part-*.parquet")))]
    select = ", ".join(f"hx({c}) AS {c}" if hex and c in HASH_COLUMNS[name] else c for c in cols)
    query = f"SELECT {select} FROM {name}"
    if where:
        query += f" WHERE {where}"
    if limit is not None:
        query += f" LIMIT {int(limit)}"
    return sql(query)


class Tables(Mapping):
    """Every table as a DataFrame, read on first access and cached."""

    def __init__(self, hex: bool = True):
        self.hex = hex
        self._cache: dict[str, pd.DataFrame] = {}

    def __getitem__(self, name: str) -> pd.DataFrame:
        if name not in self._cache:
            self._cache[name] = load(name, hex=self.hex)
        return self._cache[name]

    def __getattr__(self, name: str) -> pd.DataFrame:
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name) from None

    def __dir__(self):
        return [*super().__dir__(), *TABLES]

    def __iter__(self):
        return iter(TABLES)

    def __len__(self):
        return len(TABLES)

    def drop(self, name: str) -> None:
        """Free a cached table."""
        self._cache.pop(name, None)

    def rows(self) -> pd.Series:
        """Row counts from Parquet metadata, without reading any data."""
        return pd.Series(
            {n: con.sql(f"SELECT sum(num_rows) FROM parquet_file_metadata('{DATA / n}/part-*.parquet')").fetchall()[0][0] for n in TABLES},
            name="rows",
        )

    def __repr__(self):
        return f"Tables({len(TABLES)} tables at {DATA}; loaded: {', '.join(self._cache) or 'none'})"


tables = Tables(hex=True)
tables

In [ ]:
tables.rows().sort_values(ascending=False)

In [ ]:
tables.tx.head()

In [ ]:
# Part of a large table.
load("tx_output", columns=["tx_hash", "output_index", "address", "lovelace"], where="lovelace > 10e12").head()

In [ ]:
# Aggregate in SQL instead of loading: blocks per software tag in the header's protocol minor version
# (69 in the low byte is Dingo; see the mithril2pq README for the encoding).
sql("""
    SELECT protocol_minor & 255 AS impl_id, count(*) AS blocks, count(DISTINCT issuer_vkey) AS pools
    FROM block_header
    WHERE protocol_minor IS NOT NULL
    GROUP BY ALL ORDER BY blocks DESC
""")